# Bézier Curve — Matrix Method
### $\mathbf{P}(u) = \mathbf{U} \cdot \mathbf{M} \cdot \mathbf{G}$

All arithmetic is **exact** (SymPy rational numbers).  
No floating-point rounding anywhere in the computation.

| Symbol | Shape | Meaning |
|--------|-------|---------|
| $\mathbf{U}(u)$ | $1 \times (n+1)$ | Parameter vector $[u^n,\ u^{n-1},\ \dots,\ u,\ 1]$ |
| $\mathbf{M}$ | $(n+1) \times (n+1)$ | Basis matrix — derived from Bernstein definition |
| $\mathbf{G}$ | $(n+1) \times d$ | Geometry matrix — control points as rows |

**Derivation of $M$:**  
Each column $i$ holds the monomial coefficients of $B(n,i,u)$:
$$B(n,i,u) = \binom{n}{i}\, u^i\,(1-u)^{n-i}$$
Expanding $(1-u)^{n-i}$ via the binomial theorem gives:
$$M[n-k,\, i] = \binom{n}{i}\binom{n-i}{k-i}(-1)^{k-i} \quad \text{for } k \geq i, \text{ else } 0$$

In [1]:
import sympy as sp
from IPython.display import display, Math, Markdown

sp.init_printing(use_latex='mathjax')

# ── Case Data ────────────────────────────────────────────────

CASE_DATA = {
    1: dict(
        label    = 'Quadratic — 2D',
        points   = [[1, 1], [3, 3], [5, 1]],
        axes     = ['x', 'y'],
        u_values = [0, sp.Rational(1,4), sp.Rational(1,2),
                       sp.Rational(3,4), 1],
    ),
    2: dict(
        label    = 'Quartic — 3D',
        points   = [[0,0,1], [0,4,1], [4,0,1], [4,4,1], [5,4,1]],
        axes     = ['x', 'y', 'z'],
        u_values = [0, sp.Rational(1,4), sp.Rational(1,2),
                       sp.Rational(3,4), 1],
    ),
}

# ── BezierMatrix class ───────────────────────────────────────

class BezierMatrix:
    """
    Evaluates a Bézier curve of any degree via P(u) = U · M · G.
    All matrices use SymPy — exact rational arithmetic throughout.

    M[n-k, i] = C(n,i) · C(n-i, k-i) · (-1)^(k-i)  for k >= i
              = 0                                      for k <  i
    Row 0 = highest power u^n,  last row = constant u^0.
    """

    u = sp.Symbol('u')

    def __init__(self, points: list, axes: list):
        self.axes   = axes
        self.n      = len(points)
        self.degree = self.n - 1
        self.dim    = len(axes)

        self.G  = sp.Matrix([[sp.Rational(c) for c in row]
                              for row in points])
        self.M  = self._derive_M()
        self.MG = self.M * self.G

    def _derive_M(self) -> sp.Matrix:
        n, size = self.degree, self.n
        M = sp.zeros(size, size)
        for i in range(size):
            m = n - i
            for j in range(m + 1):
                k = i + j
                M[n-k, i] = sp.binomial(n,i) * sp.binomial(m,j) * (-1)**j
        return M

    def U(self, u_val) -> sp.Matrix:
        """Row vector [u^n, u^(n-1), ..., u, 1]."""
        return sp.Matrix([[u_val**(self.degree - k)
                           for k in range(self.degree + 1)]])

    def evaluate_all(self, u_values: list) -> sp.Matrix:
        """Evaluate at all u values — one batched matrix multiply."""
        U_all = sp.Matrix([[u**(self.degree - k)
                            for k in range(self.degree + 1)]
                           for u in u_values])
        return U_all * self.MG

    def poly_expr(self, axis_idx: int):
        """Return the monomial polynomial for one axis as a SymPy expr."""
        coeffs = [self.MG[k, axis_idx] for k in range(self.n)]
        return sp.Poly(coeffs, self.u).as_expr()

---
## Switch
Change `CASE` to `1` (Quadratic 2D) or `2` (Quartic 3D).

In [2]:
CASE = 1   # ← change here: 1 = Quadratic 2D,  2 = Quartic 3D

data = CASE_DATA[CASE]
bm   = BezierMatrix(points=data['points'], axes=data['axes'])

names = {2:'Quadratic', 3:'Cubic', 4:'Quartic', 5:'Quintic'}
display(Markdown(
    f"### Case {CASE} — {names[bm.degree]} | {bm.dim}D"
    f" | {bm.n} Control Points"
))

### Case 1 — Quadratic | 2D | 3 Control Points

---
## $\mathbf{G}$ — Geometry Matrix (Control Points)

In [3]:
# Label columns with axis names, rows with P0..Pn
axes_latex = ',\\, '.join(data['axes'])
display(Math(r'\mathbf{G} = ' + sp.latex(bm.G) +
             r'\quad \text{columns: }' + axes_latex))

<IPython.core.display.Math object>

---
## $\mathbf{M}$ — Basis Matrix

Derived automatically from the Bernstein definition.  
Each column $i$ holds the monomial coefficients of $B(n,i,u)$.

In [4]:
display(Math(r'\mathbf{M} = ' + sp.latex(bm.M)))

<IPython.core.display.Math object>

---
## $\mathbf{M} \cdot \mathbf{G}$ — Polynomial Coefficient Matrix

Row $k$ of $\mathbf{MG}$ is the coefficient of $u^{n-k}$ in $P(u)$.

In [5]:
display(Math(r'\mathbf{M} \cdot \mathbf{G} = ' + sp.latex(bm.MG)))

<IPython.core.display.Math object>

---
## Resulting Polynomials

Reading the columns of $\mathbf{MG}$ directly gives each component polynomial.

In [6]:
for idx, axis in enumerate(bm.axes):
    expr = bm.poly_expr(idx)
    display(Math(f'{axis}(u) = ' + sp.latex(expr)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

---
## Evaluation Table — $P(u) = \mathbf{U}(u) \cdot \mathbf{MG}$

Exact rational results from one batched matrix multiply.

In [7]:
pts = bm.evaluate_all(data['u_values'])

# Build a SymPy matrix with u column prepended for display
u_col  = sp.Matrix(data['u_values'])
table  = u_col.row_join(pts)

# Header row as a separate matrix for labelling
header = ['u'] + [f'{ax}(u)' for ax in bm.axes]

# Display with decimal equivalents underneath
display(Math(r'\begin{array}{' + 'r'*(bm.dim+1) + r'}'
    + ' & '.join(f'\\mathbf{{{h}}}' for h in header)
    + r' \\ \hline '
    + r' \\ '.join(
        ' & '.join(
            sp.latex(table[r, c]) for c in range(table.shape[1])
        )
        for r in range(table.shape[0])
    )
    + r'\end{array}'))

# Decimal approximations
print('\nDecimal approximations:')
for r, u_val in enumerate(data['u_values']):
    vals = '   '.join(
        f'{bm.axes[c]}={float(pts[r,c]):.4f}'
        for c in range(bm.dim)
    )
    print(f'  u={float(u_val):.2f}   {vals}')

<IPython.core.display.Math object>


Decimal approximations:
  u=0.00   x=1.0000   y=1.0000
  u=0.25   x=2.0000   y=1.7500
  u=0.50   x=3.0000   y=2.0000
  u=0.75   x=4.0000   y=1.7500
  u=1.00   x=5.0000   y=1.0000
